In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os, re, sys, json
import pandas as pd
from requests.models import Response
from datetime import datetime as dt 
from dateutil.relativedelta import relativedelta
from rockyelevate.wrapper import Session
from rockyelevate.utils import response_to_dataframe
from rockyclickup.models import MODEL_LOOKUP, Task
from rockyclickup.database_interface import get_all_fields
from rockyclickup.wrapper import Session as rcu

with open(rf"G:\Shared drives\engine\.assets\annual_election_irs_max.json", 'r') as file:
    IRS_MAX_ELECTION_MAP = json.load(file)

current_dir = os.getcwd()
parent_dir = os.path.dirname(f"{"\\".join(current_dir.split("\\")[:-1])}")
sys.path.append(parent_dir)

from constants.maps import (
    ELV_TEMPLATE_IDS,
    ELV_STATUS_MAP,
    FIELDS_TO_POP,
    FIELDS_TO_COPY,
    FIELDS_TO_SET,
    ELV_ACCOUNT_TYPE_MAP
)

# from engine.apps.meridian.constants import ELV_TEMPLATE_IDS, ELV_STATUS_MAP, FIELDS_TO_POP, FIELDS_TO_COPY, FIELDS_TO_SET

today = dt.now()
elv = Session("PROD", multithread=True, max_threads=40)
clickup = rcu()

In [13]:
''' load plans from previous notebook ''' # ./filter.ipynb

try:
    plan_df = pd.read_pickle(f"cache/{dt.strftime(today, "%y%m%d")}_FILTERED_PLANS.pkl")
    print(f"opened {len(plan_df)} plans")
except FileNotFoundError as e:
    print(f"{e}, please generate file in `filter.ipynb`")

opened 113 plans


In [14]:
''' filter df to rows without errors '''

display(plan_df['error'].value_counts())
no_errors = plan_df[plan_df['error'].apply(lambda x: len(x) == 0)]
print(f"{len(no_errors)} plans without errors")

error
[]                                                     70
[future plan exists]                                   27
[skip orgs with complex HRA's]                          6
[non-active organization]                               5
[auto-renew not turned on, non-active organization]     2
[skip orgs with complex HRA's, future plan exists]      2
[short plan year]                                       1
Name: count, dtype: int64

70 plans without errors


In [15]:
''' roll valid from/to '''

new_plan_data = no_errors.copy()
new_plan_data['new_valid_from'] = new_plan_data['plan_year.valid_from'].apply(lambda x: x + relativedelta(years=1))
new_plan_data['new_valid_to'] = new_plan_data['plan_year.valid_to'].apply(lambda x: x + relativedelta(years=1))

In [26]:

''' map plan year id'''
needed_plan_years = new_plan_data[
    [
        'organization_id',
        'plan_year_id',
        'new_valid_from',
        'new_valid_to',
    ]
]\
    .drop_duplicates()\
    .sort_values("new_valid_from")

# fetch existing plan years for relevant organizations
org_ids = needed_plan_years['organization_id'].unique()
plan_year_res = elv.get_plan_years(oids=org_ids)

# create a map from existing plan years
# keys are prior plan year ids and values are the actual plan year id
prior_plan_year_map = {
    res.get("prior_plan_year_id"): res.get("id")
    for res in plan_year_res
}
new_plan_data['new_plan_year_id'] = new_plan_data['plan_year_id'].map(prior_plan_year_map)
print(f"{len(new_plan_data[new_plan_data['new_plan_year_id'].isna()])} rows are missing new plan year ids")

''' map template ids '''
new_plan_data['template_id'] = new_plan_data['account_type.account_type'].map(ELV_TEMPLATE_IDS.get("PROD"))
print(f"{len(new_plan_data[pd.isna(new_plan_data['template_id'])])} rows are missing template ids")

''' new plan code '''
def generate_plan_code(row):

    normalized_account_type = ELV_ACCOUNT_TYPE_MAP.get(row.get('account_type.account_type'))

    if any([
        pd.isna(row.get("new_valid_from")),
        pd.isna(row.get("new_valid_to")),
        pd.isna(row.get("rmrcode")),
        pd.isna(normalized_account_type),
    ]):
        raise ValueError(f"""
            Missing properties:\n
            \tnew_valid_from: {row.get("new_valid_from")}
            \tnew_valid_to: {row.get("new_valid_to")}
            \trmrcode: {row.get("rmrcode")}
            \tcu_account_type: {normalized_account_type}
        """)

    return f"{row.get("rmrcode")}{normalized_account_type}{dt.strftime(row.get("new_valid_from"), "%m%d%Y")}{dt.strftime(row.get("new_valid_to"), "%m%d%Y")}"

new_plan_data['new_plan_code'] = new_plan_data.apply(generate_plan_code, axis=1)
print(f"{len(new_plan_data[pd.isna(new_plan_data['new_plan_code'])])} rows are missing new plan codes")

''' new plan name '''
def generate_plan_name(row):

    # get current plan name
    new_plan_name = row['elv_plan_name']
    # finnd all numbers in plan name
    number_substrings = re.findall(r'\d+', new_plan_name)

    # loop through all numbers
    for num in number_substrings:
        # if the number is larger than `2020` remove it
        if int(num) > 2020:
            new_plan_name = new_plan_name.replace(num, "")

    # split the name on spaces and rejoin.
    # forgot why i did this, prolly to fix formatting issues
    name_split = new_plan_name.split(" ")
    name_split = [s for s in name_split if s != '']
    new_plan_name = " ".join(name_split)

    # return early if it's an hsa
    if (pd.notna(row['account_type.account_type'].lower()) and row['account_type.account_type'].lower() == "hsa") or (pd.notna(row['cu_account_type']) and row['cu_account_type'].lower() == "hsa"):
        return new_plan_name

    # otherwise add the current plan year to the new plan name
    new_plan_name = f"{new_plan_name} {row['new_valid_from'].year}"
    return new_plan_name

new_plan_data['new_plan_name'] = new_plan_data.apply(generate_plan_name, axis=1)
print(f"{len(new_plan_data[pd.isna(new_plan_data['new_plan_name'])])} rows are missing new plan names")

''' generate naked bodies to post to elevate '''
def generate_naked_body(row):
    return {
        "organization_id": row.get("organization_id"),
        "plan_code": row.get("new_plan_code"),
        "name": {
            "name": row.get("new_plan_name"),
            "name_state": "MODIFIABLE"
        },
        "prior_plan_id": row.get("elv_plan_id"), 
        "parent_id": row.get("template_id"),
        "plan_year_id": row.get("new_plan_year_id"),
        "is_plan": True,
    }
new_plan_data['naked_body'] = new_plan_data.apply(generate_naked_body, axis=1)

print(f"{len(new_plan_data[pd.isna(new_plan_data['naked_body'])])} rows are missing naked bodies")

0 rows are missing new plan year ids
0 rows are missing template ids
0 rows are missing new plan codes
0 rows are missing new plan names
0 rows are missing naked bodies


In [27]:
new_plan_data.to_csv("plans_before_sending_naked_bodes.csv")

In [25]:
''' create needed plan years '''
''' AFTER RUNNING THIS CELL, RUN THE CELL ABOVE TO COMPLETE NAKED BODIES '''

plan_year_endpoint = "https://api.prod.elevateaccounts.com/v1/plan-years"

missing_plan_years = new_plan_data[pd.isna(new_plan_data['new_plan_year_id'])]

# print(len(missing_plan_years))

for index, row in missing_plan_years.iterrows():
    
    valid_from_str = dt.strftime(row.get("new_valid_from"), "%m/%d/%Y")
    valid_to_str = dt.strftime(row.get("new_valid_to"), "%m/%d/%Y")

    print(type(row.get("plan_year_id")))


    new_plan_year_body = {
        "organization_id": row.get("organization_id"),
        "name": f"{valid_from_str} - {valid_to_str}",
        "valid_from": valid_from_str,
        "valid_to": valid_to_str,
        "prior_plan_year_id": row.get("plan_year_id")
    }

    display(new_plan_year_body)

    new_plan_year_res = elv.post(
        endpoint=plan_year_endpoint,
        payload=new_plan_year_body
    )

    # organizaiton id
    # plan year name
    # valid from
    # valid to
    # prior plan year id

<class 'int'>


{'organization_id': 8717,
 'name': '11/01/2025 - 10/31/2026',
 'valid_from': '11/01/2025',
 'valid_to': '10/31/2026',
 'prior_plan_year_id': 32220}

<class 'int'>


{'organization_id': 8717,
 'name': '11/01/2025 - 10/31/2026',
 'valid_from': '11/01/2025',
 'valid_to': '10/31/2026',
 'prior_plan_year_id': 32220}

<class 'int'>


...

{'organization_id': 10863,
 'name': '11/01/2025 - 10/31/2026',
 'valid_from': '11/01/2025',
 'valid_to': '10/31/2026',
 'prior_plan_year_id': 38789}

<class 'int'>


{'organization_id': 10863,
 'name': '11/01/2025 - 10/31/2026',
 'valid_from': '11/01/2025',
 'valid_to': '10/31/2026',
 'prior_plan_year_id': 38789}

<class 'int'>


{'organization_id': 10863,
 'name': '11/01/2025 - 10/31/2026',
 'valid_from': '11/01/2025',
 'valid_to': '10/31/2026',
 'prior_plan_year_id': 38789}

---
### <b> SEND NAKED BODIES </b>
---

In [28]:
''' send naked bodies to elevate ''' # ~25s

new_plan_data['post_naked_err'] = None
url = f"{elv.basepath}/plans"
successful_plan_ids = []
for index, row in new_plan_data.iterrows():
    naked_body = row['naked_body']

    # send naked body to elevate
    try:
        post_naked_res = elv.post(url, payload=naked_body)

    except Exception as e:
        err_msg = f"Error posting naked plan to Elevate: {e}"
        new_plan_data.loc[row.name, 'post_naked_err'] = err_msg
        print(f"Request failed for index {index}: {err_msg}")
        continue
    
    if isinstance(post_naked_res, tuple):
        response = post_naked_res[1]

        # success
        if response.status_code >= 200 and response.status_code < 300:
            try:
                post_naked_json = response.json()
                successful_plan_ids.append(post_naked_json.get('id'))

            except ValueError as json_error:
                err_msg = f"Failed to parse JSON response: {json_error}"
                new_plan_data.loc[row.name, 'post_naked_err'] = err_msg
                print(err_msg)

        # error
        else: 
            err_msg = f"HTTP {response.status_code}: {response.text}"
            new_plan_data.loc[row.name, 'post_naked_err'] = err_msg 
            print(f"HTTP error for index {index}: {err_msg}")

    else:
        new_plan_data.loc[row.name, 'post_naked_err'] = str(post_naked_res)
        print(f"Unexpected response format for index {index}: {type(post_naked_res)}")

print(len(successful_plan_ids))


HTTP error for index 4814: HTTP 409: {
  "elevate_error_code" : "40900",
  "elevate_error_message" : "The request could not be completed due to a conflict with the current state of the target resource. Details: Plan with plan_code 'RMRMSIHRA1201202511302026', organization_id '10167' already exists.",
  "error_at" : "2025-09-24T06:28:46.467170179Z"
}
HTTP error for index 4815: HTTP 409: {
  "elevate_error_code" : "40900",
  "elevate_error_message" : "The request could not be completed due to a conflict with the current state of the target resource. Details: Plan with plan_code 'RMRMSIHRA1201202511302026', organization_id '10167' already exists.",
  "error_at" : "2025-09-24T06:28:46.990019468Z"
}
68


In [31]:
new_plan_data.to_csv("after_posting_naked_bodies.csv")

In [30]:
new_plan_data[new_plan_data['post_naked_err'].notna()]

,elv_plan_id,elv_plan_parent_id,plan_year_id,is_plan,is_forfeited,plan_code,plan_omnibus_account_id,notional_payroll_account_id,notional_funding_account_id,elv_plan_status,...,error,warning,new_valid_from,new_valid_to,new_plan_year_id,template_id,new_plan_code,new_plan_name,naked_body,post_naked_err
4814,48044,36584,33948,True,False,RMRMSIDRA1201202411302025,1646655.0,1646656.0,1646657.0,ACTIVE,...,[],"[hra, no `cu_plan_id` found, org has HRA's]",2025-12-01,2026-11-30,78744,5,RMRMSIHRA1201202511302026,Dental Reimbursement Arrangement (HRA) 2025,"{'organization_id': 10167, 'plan_code': 'RMRMS...","HTTP 409: {\n ""elevate_error_code"" : ""40900"",..."
4815,48045,36585,33948,True,False,RMRMSIVRA1201202411302025,1646723.0,1646724.0,1646725.0,ACTIVE,...,[],"[hra, no `cu_plan_id` found, org has HRA's]",2025-12-01,2026-11-30,78744,5,RMRMSIHRA1201202511302026,Vision Reimbursement Arrangement (HRA) 2025,"{'organization_id': 10167, 'plan_code': 'RMRMS...","HTTP 409: {\n ""elevate_error_code"" : ""40900"",..."


In [33]:
''' get and add new plan information to df ''' # ~3s
org_ids = new_plan_data['organization_id'].unique()
org_plans = elv.get_plans_by_org(org_ids, detail=True)

new_plans = [p for p in org_plans if p.get("prior_plan_id") in new_plan_data['elv_plan_id'].to_list()]
prior_plans = [p for p in org_plans if p.get("id") in new_plan_data['elv_plan_id'].to_list()]


new_plan_res_map = {
    p.get("id"): p
    for p in new_plans
}

prior_plan_id_map = {
    p.get("prior_plan_id"): p.get('id')
    for p in new_plans
}

prior_plan_res_map = {
    p.get("id"): p
    for p in prior_plans
}

new_plan_data['new_elv_plan_id'] = new_plan_data['elv_plan_id'].map(prior_plan_id_map)
new_plan_data['new_plan_res'] = new_plan_data['new_elv_plan_id'].map(new_plan_res_map)

---
### <b> UPDATE NAKED BODIES </B>
---

In [35]:
''' update plans on elevate ''' # ~30s

updated_plan_responses = []
for index, row in new_plan_data.iterrows():
    if pd.isna(row['new_plan_res']):
        continue

    updated_plan_res = row['new_plan_res'].copy()
    prior_plan_res = prior_plan_res_map.get(row['elv_plan_id'])

    # remove unnecessary fields from new plan res
    for field in [f for f in FIELDS_TO_POP if f in updated_plan_res]:
        updated_plan_res.pop(field, None)
    
    # copy fields from prior plan to new plan
    for field in [f for f in FIELDS_TO_COPY if f in prior_plan_res]:
        updated_plan_res[field] = prior_plan_res[field]

    # set hardcoded field values
    for field, value in FIELDS_TO_SET.items():
        updated_plan_res[field] = value
        
    elv_update_response = elv.update_plan(pid=updated_plan_res.get('id'), data=updated_plan_res)
    updated_plan_responses.append(updated_plan_res)

updated_response_map = {
    p.get('id'): p
    for p in updated_plan_responses
}

new_plan_data['new_plan_res'] = new_plan_data['new_elv_plan_id'].map(updated_response_map)


In [37]:
new_plan_data.to_csv("after_update_naked_bodies.csvs")

---

'list_id'


KeyError: 'list_id'

In [ ]:
am_field_id = db_field_map.get('account_manager').field_id

new_plan_data['new_clickup_plan_id'] = None
new_plan_data['clickup_res'] = None 
new_plan_data['clickup_err'] = None

new_plan_data['link_am_res'] = None
new_plan_data['link_client_res'] = None

for index, row in new_plan_data.iterrows():
    model = row['rcu_model']

    cu_create_res = clickup.create(model)

    if isinstance(cu_create_res, Response):
        if 200 <= cu_create_res.status_code < 300:
            res_json = cu_create_res.json()
            new_task_id = res_json.get('id')


            new_plan_data.loc[index, 'new_clickup_plan_id'] = new_task_id
            new_plan_data.loc[index, 'clickup_res'] = str(res_json)

            account_manager_ids = row['cu_plan_account_manager']
            if isinstance(account_manager_ids, int):
                account_manager_ids = [account_manager_ids]

            if account_manager_ids:
                link_am_res = clickup.patch(
                    new_task_id,
                    am_field_id,
                    add=account_manager_ids
                )

                new_plan_data.loc[index, 'link_am_res'] = str(link_am_res)

            client_ids = None
            client_id = row['client_id']

            if isinstance(client_id, str):
                client_ids = [client_id]
            if isinstance(client_id, list):
                client_ids = client_id
            if client_ids:
                relation_field_id = db_field_map.get(f"client_{model.category}").field_id
                link_client_res = clickup.patch(
                    new_task_id,
                    relation_field_id,
                    add=client_ids
                )

                new_plan_data.loc[index, 'link_client_res'] = str(link_client_res)

        else:
            new_plan_data['clickup_err'] = str(cu_create_res.text)

    else: 
        new_plan_data['clickup_err'] = str(cu_create_res)